In [3]:
import os
import sys
# add the 'src' directory as one where we can import modules
PROJ_ROOT = os.path.join(os.pardir)
src_dir = os.path.join(PROJ_ROOT, 'src')
sys.path.append(src_dir)

print(os.getcwd())
print(sys.path)

import IPython.extensions.autoreload
%load_ext autoreload
%autoreload 2

# import my method from the source code
from ingest import data_workflow
from dataset import analyze, detect_elapsed_time_anomalies, resample
import matplotlib.pyplot as plt

c:\Users\Loris Amabile\Documents\c_market\notebooks
['C:\\Users\\Loris Amabile\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\python312.zip', 'C:\\Users\\Loris Amabile\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\DLLs', 'C:\\Users\\Loris Amabile\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib', 'C:\\Users\\Loris Amabile\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none', 'c:\\Users\\Loris Amabile\\Documents\\c_market\\.venv', '', 'c:\\Users\\Loris Amabile\\Documents\\c_market\\.venv\\Lib\\site-packages', 'c:\\Users\\Loris Amabile\\Documents\\c_market\\.venv\\Lib\\site-packages\\win32', 'c:\\Users\\Loris Amabile\\Documents\\c_market\\.venv\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\Loris Amabile\\Documents\\c_market\\.venv\\Lib\\site-packages\\Pythonwin', '..\\src', '..\\src']
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
df = data_workflow("inariz")


# include the unit into the main column name
df = df.rename(columns={"Valeur": "steam_production_(m3/h)"})

# for other columns, keep only numerical columns
# set the timestamp as a column with a standard name, not as the index
df = df.drop(columns=["Unité"])


analyze(df)
print(df)
elapsed_anomalies, expected_delta = detect_elapsed_time_anomalies(df, timestamp_col="measured_at_utc")
# target
df = df[["measured_at_utc", "steam_production_(m3/h)"]] #todo check, useless je pense (et faux ?)
df_10min = resample(df, desired_timedelta="10min", aggregate_function="mean")
df_15min = resample(df, desired_timedelta="15min", aggregate_function="mean")

                  measured_at      Valeur Unité
0     2025-12-15 00:00:00.000  226.403839  m3/h
1     2025-12-15 00:01:16.004  328.005280  m3/h
2     2025-12-15 00:01:36.009  223.755951  m3/h
3     2025-12-15 00:02:02.016  324.902985  m3/h
4     2025-12-15 00:03:36.016  220.323868  m3/h
...                       ...         ...   ...
37054 2025-12-20 08:19:12.005  573.057434  m3/h
37055 2025-12-20 08:19:23.004  471.931213  m3/h
37056 2025-12-20 08:19:27.005  581.450562  m3/h
37057 2025-12-20 08:19:29.001  692.254395  m3/h
37058 2025-12-20 08:19:30.018  802.520569  m3/h

[37059 rows x 3 columns]


Verify that the average is done correctly between [00:00; 00:15[

In [ ]:
import datetime 
df_15min[(df_15min["measured_at_utc"] >= datetime.datetime(2025, 12, 15, 0, 0, 0)) & (df_15min["measured_at_utc"] <= datetime.datetime(2025, 12, 15, 1, 0, 0))]

TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and datetime

In [ ]:
df[(df["measured_at_utc"] >= datetime.datetime(2025, 12, 15, 0, 15, 0)) & (df["measured_at_utc"] < datetime.datetime(2025, 12, 15, 0, 30, 0))]["steam_production_(m3/h)"].describe()

In [ ]:
# plot original and resampled timeseries, to check the choice of aggregate_function
import PyQt6
%matplotlib qt
fig, ax = plt.subplots()

ax.plot(df["measured_at_utc"], df["steam_production_(m3/h)"], label="original")
ax.step(df_10min["measured_at_utc"], df_10min["steam_production_(m3/h)"], label="resampled 10 min", where="post")
ax.step(df_15min["measured_at_utc"], df_15min["steam_production_(m3/h)"], label="resampled 15 min", where="post")
ax.set_ylabel("Value")
# ax.set_title(path.name)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

Try simplest forms of prediction

In [ ]:
from predict import simple_copy, copy_median_values
import datetime

# df_copy1day = simple_copy(df_15min, "measured_at_utc", "steam_production_(m3/h)", datetime.date(2025,12,15), datetime.date(2025,12,16))
# df_copy1week = simple_copy(df_15min, "measured_at_utc", "steam_production_(m3/h)", datetime.date(2025,12,15), datetime.date(2025,12,20))

df_median1day = copy_median_values(df_15min, "measured_at_utc", "steam_production_(m3/h)", use_holidays=False, use_weekdays=True, use_time=True, extension="jour")
# df_median1week = copy_median_values(y_15min, "measured_at_utc", "steam_production_(m3/h)", use_holidays=False, use_weekdays=True, use_time=True, extension="semaine")


TEMP verification pour simple_copy

In [ ]:
from datetime import tzinfo
start = datetime.datetime(2025,12,15)
start = pd.to_datetime(start).tz_localize("Europe/Paris").tz_convert("UTC")
start

Timestamp('2025-12-14 23:00:00+0000', tz='UTC')

In [ ]:
df_copy1day[df_copy1day["measured_at_utc"]==start]

,measured_at,steam_production_(m3/h)
0,2025-12-14 23:00:00+00:00,675.168389


In [ ]:
df_copy1day.iloc[-100:-90].measured_at_utc.dt.weekday

511    5
512    5
513    5
514    5
515    5
516    5
517    5
518    6
519    6
520    6
Name: measured_at, dtype: int32

In [ ]:
# df_copy1week[df_copy1week.measured_at_utc == start_new_1week]
# df_copy1week.iloc[len(df_15min)-3:len(df_15min)+3]
print(df_copy1week.iloc[len(df_15min)].measured_at_utc.weekday())
start  = datetime.date(2025,12,15)
start = pd.to_datetime(start, errors="coerce")
start = start.tz_localize(source_timezone)
start = start.tz_convert("UTC")

print(df_15min[df_15min.measured_at_utc == start].measured_at_utc.dt.weekday)


6
0    6
Name: measured_at, dtype: int32


temp vérification copy median

In [103]:
y_median1day.iloc[len(y_15min)-3:len(y_15min)+3]

,measured_at,steam_production_(m3/h)
511,2025-12-20 06:45:00+00:00,530.893062
512,2025-12-20 07:00:00+00:00,533.215119
513,2025-12-20 07:15:00+00:00,832.717848
514,2025-12-20 07:30:00+00:00,682.966483
515,2025-12-20 07:45:00+00:00,682.966483
516,2025-12-20 08:00:00+00:00,582.585169


copy_median_value copie forcément juste après la fin du df inial
ce que je dois vérifier, c'est si, pour faire la valeur du 2025-12-20 07:30:00+00:00, il a bien pris la mediane des jours et weekdays et créneaux pertinents.